# StyleGAN2-ADA Latent Space Manipulation

In [ ]:
# Cell 1 - GPU Kontrolü ve Kurulum
import torch
import os
import sys

print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device Name: {torch.cuda.get_device_name(0)}")
else:
    print("UYARI: GPU aktif değil! Kaggle ayarlarından 'Accelerator' kısmını GPU yap.")

if not os.path.exists('stylegan2-ada-pytorch'):
    os.system('git clone https://github.com/NVlabs/stylegan2-ada-pytorch.git')

os.system('pip install ninja -q')
sys.path.insert(0, "/kaggle/working/stylegan2-ada-pytorch")
print("Kurulum tamamlandı.")

In [ ]:
# Cell 2 - Model Yükleme
import dnnlib
import legacy
import torch

device = torch.device('cuda')
network_pkl = 'https://nvlabs-fi-cdn.nvidia.com/stylegan2-ada-pytorch/pretrained/ffhq.pkl'

print(f'Model yükleniyor: {network_pkl} ...')
with dnnlib.util.open_url(network_pkl) as f:
    G = legacy.load_network_pkl(f)['G_ema'].to(device)

print(f"Model yüklendi. W boyutu: {G.w_dim}, Z boyutu: {G.z_dim}")

In [ ]:
# Cell 3 - Test Görüntüsü
import PIL.Image
import matplotlib.pyplot as plt
import numpy as np

def generate_image(G, z, truncation_psi=0.7):
    #Z vektöründen tek görüntü üretir.
    with torch.no_grad():
        img = G(z, None, truncation_psi=truncation_psi, noise_mode='const')
    img = (img.permute(0, 2, 3, 1) * 127.5 + 128).clamp(0, 255).to(torch.uint8)
    return PIL.Image.fromarray(img[0].cpu().numpy(), 'RGB')

seed = 42
torch.manual_seed(seed)
z = torch.randn([1, G.z_dim]).to(device)
img = generate_image(G, z)

plt.figure(figsize=(6, 6))
plt.imshow(img)
plt.axis('off')
plt.title(f"Test Görüntüsü (Seed: {seed})")
plt.show()

In [ ]:
# Cell 4 - Classifier Eğitimi

import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
import torchvision.models as models
from torchvision.models import ResNet18_Weights  
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import f1_score             
import pandas as pd
from PIL import Image
import os
from tqdm import tqdm

TARGET_ATTR = 'Smiling'
BATCH_SIZE  = 64
EPOCHS      = 3
LR          = 0.001
DEVICE      = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

ATTR_PATH = '/kaggle/input/celeba-dataset/list_attr_celeba.csv'
IMG_DIR   = '/kaggle/input/celeba-dataset/img_align_celeba/img_align_celeba'


class CelebADataset(Dataset):
    def __init__(self, attr_path, img_dir, target_attr, transform=None, limit=30000):
        self.attr_df     = pd.read_csv(attr_path).iloc[:limit]
        self.img_dir     = img_dir
        self.transform   = transform
        self.target_attr = target_attr
        self.attr_df[self.target_attr] = (self.attr_df[self.target_attr] + 1) // 2

    def __len__(self):
        return len(self.attr_df)

    def __getitem__(self, idx):
        row      = self.attr_df.iloc[idx]
        img_name = row['image_id']
        img_path = os.path.join(self.img_dir, img_name)

        try:
            image = Image.open(img_path).convert('RGB')
        except (FileNotFoundError, OSError):
            parent_dir = os.path.dirname(self.img_dir)
            img_path   = os.path.join(parent_dir, img_name)
            image      = Image.open(img_path).convert('RGB')

        label = torch.tensor(row[self.target_attr], dtype=torch.float32)
        if self.transform:
            image = self.transform(image)
        return image, label


transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.ToTensor(),
    transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5])
])

print("Veri seti hazırlanıyor...")
dataset      = CelebADataset(ATTR_PATH, IMG_DIR, TARGET_ATTR, transform)
train_size   = int(0.8 * len(dataset))
val_size     = len(dataset) - train_size
train_ds, val_ds = torch.utils.data.random_split(dataset, [train_size, val_size])

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=4, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)

print("Model hazırlanıyor...")
classifier     = models.resnet18(weights=ResNet18_Weights.DEFAULT)
classifier.fc  = nn.Linear(classifier.fc.in_features, 1)
classifier     = classifier.to(DEVICE)

criterion  = nn.BCEWithLogitsLoss()
optimizer  = optim.Adam(classifier.parameters(), lr=LR)

print(f"Eğitim başlıyor... ({EPOCHS} epoch) - Hedef: {TARGET_ATTR}")
for epoch in range(EPOCHS):
    classifier.train()
    running_loss = 0.0
    loop = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}")

    for images, labels in loop:
        images, labels = images.to(DEVICE), labels.to(DEVICE).unsqueeze(1)
        optimizer.zero_grad()
        outputs = classifier(images)
        loss    = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        loop.set_postfix(loss=f"{loss.item():.4f}")

    classifier.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE).unsqueeze(1)
            outputs        = classifier(images)
            predicted      = (torch.sigmoid(outputs) > 0.5).float()
            all_preds.append(predicted.cpu())
            all_labels.append(labels.cpu())

    all_preds  = torch.cat(all_preds).numpy().flatten()
    all_labels = torch.cat(all_labels).numpy().flatten()
    acc = 100 * (all_preds == all_labels).mean()
    f1  = f1_score(all_labels, all_preds)
    print(f"--- Epoch {epoch+1} | Val Acc: {acc:.2f}% | F1: {f1:.3f} ---")

torch.save(classifier.state_dict(), 'classifier_resnet18.pth')
print("Classifier eğitimi tamamlandı!")

In [ ]:
# Cell 5 - Latent Space Veri Toplama

import torch
import torch.nn.functional as F
import numpy as np
from tqdm import tqdm

NUM_SAMPLES = 5000
BATCH_SIZE  = 16

zs, ws, labels, scores = [], [], [], []

print(f"{NUM_SAMPLES} adet örnek üretiliyor ve etiketleniyor...")
classifier.eval()

with torch.no_grad():
    for i in tqdm(range(0, NUM_SAMPLES, BATCH_SIZE)):
        curr_batch = min(BATCH_SIZE, NUM_SAMPLES - i)
        if curr_batch <= 0:
            break

        z = torch.randn([curr_batch, G.z_dim]).to(device)
        w = G.mapping(z, None)   # [B, 18, 512]

        img_raw = G.synthesis(w, noise_mode='const')   # [-1, 1]

        img_for_cls = F.interpolate(
            img_raw, size=(256, 256),
            mode='bilinear', align_corners=False
        )  # GAN çıktısı [-1,1]

        logits = classifier(img_for_cls)
        probs  = torch.sigmoid(logits).squeeze(1)      # [B]
        preds  = (probs > 0.5).float()

        zs.append(z.cpu().numpy())
        ws.append(w[:, 0, :].cpu().numpy())   # W

        labels.append(preds.cpu().numpy())
        scores.append(probs.cpu().numpy())    # continuous [0..1]

zs     = np.concatenate(zs,     axis=0)
ws     = np.concatenate(ws,     axis=0)
labels = np.concatenate(labels, axis=0).flatten()
scores = np.concatenate(scores, axis=0).flatten()

print(f"\nVeri toplama bitti!")
print(f"Z shape     : {zs.shape}")
print(f"W shape     : {ws.shape}")
print(f"Labels      : {labels.shape}  (pos: {int(labels.sum())}, neg: {int((1-labels).sum())})")
print(f"Score range : {scores.min():.3f} – {scores.max():.3f}")

In [ ]:
# Cell 6 - Yön Vektörü Hesaplama

from sklearn.svm import LinearSVC
import numpy as np


def get_direction_mean_diff(vectors, labels):
    #Pozitif ve negatif sınıf merkezlerinin farkı.
    pos = vectors[labels == 1]
    neg = vectors[labels == 0]
    direction = pos.mean(axis=0) - neg.mean(axis=0)
    return direction / np.linalg.norm(direction)


def get_direction_svm(vectors, labels):
    #LinearSVC hyperplane normal vektörü.
    clf = LinearSVC(max_iter=10000, dual='auto')
    clf.fit(vectors, labels)
    direction = clf.coef_.flatten()
    return direction / np.linalg.norm(direction)


def get_direction_score_weighted(vectors, scores):
    weights = np.abs(scores - 0.5) * 2   # [0, 1]

    pos_mask = scores > 0.5
    neg_mask = scores <= 0.5

    w_pos = weights[pos_mask]
    w_neg = weights[neg_mask]

    mean_pos = np.average(vectors[pos_mask], axis=0, weights=w_pos)
    mean_neg = np.average(vectors[neg_mask], axis=0, weights=w_neg)

    direction = mean_pos - mean_neg
    return direction / np.linalg.norm(direction)


print("Yönler hesaplanıyor...")

n_z_mean    = get_direction_mean_diff(zs, labels)
n_z_svm     = get_direction_svm(zs, labels)

n_w_mean    = get_direction_mean_diff(ws, labels)
n_w_svm     = get_direction_svm(ws, labels)
n_w_score   = get_direction_score_weighted(ws, scores) 

print("Hesaplama tamamlandı!")
print("Vektörler: n_z_mean, n_z_svm, n_w_mean, n_w_svm, n_w_score")

In [ ]:
# Cell 7 - Manipülasyon Fonksiyonu

import torch
import PIL.Image

def manipulate(G, z, direction, alpha, space='w', layer_range=None):
    """Z veya W uzayında görüntü manipülasyonu.

    Args:
        G          : StyleGAN2 generator
        z          : [1, z_dim] latent kod
        direction  : numpy array, normalize edilmiş yön vektörü
        alpha      : manipülasyon şiddeti (negatif = ters yön)
        space      : 'z' veya 'w'
        layer_range: W-space için hangi katmanlar etkilensin (None=hepsi).
                     Orta katmanlar (5-12) yüz özelliklerini kontrol eder.
    """
    if not isinstance(direction, torch.Tensor):
        n = torch.tensor(direction, dtype=torch.float32).to(device)
    else:
        n = direction.clone().detach().to(device)

    with torch.no_grad():
        if space == 'z':
            z_new = z + alpha * n
            img   = G(z_new, None, truncation_psi=0.7, noise_mode='const')

        elif space == 'w':
            w     = G.mapping(z, None)          # [1, num_layers, w_dim]
            w_new = w.clone()

            w_dim = w.shape[-1]                 # genellikle 512
            n_w   = n.view(1, 1, w_dim)         # broadcast için

            # DÜZELTME 2: seçici katman müdahalesi
            if layer_range is None:
                layer_range = range(w.shape[1])  # tüm katmanlar

            for l in layer_range:
                w_new[:, l, :] += alpha * n_w.squeeze(1)

            img = G.synthesis(w_new, noise_mode='const')

        else:
            raise ValueError(f"Geçersiz space: '{space}'. 'z' veya 'w' olmalı.")

    img = (img.permute(0, 2, 3, 1) * 127.5 + 128).clamp(0, 255).to(torch.uint8)
    return PIL.Image.fromarray(img[0].cpu().numpy(), 'RGB')


print("Manipülasyon fonksiyonu hazır.")

In [ ]:
# Cell 8 - Z-Space vs W-Space Karşılaştırması
import matplotlib.pyplot as plt

seed = 106
torch.manual_seed(seed)
z_sample = torch.randn([1, G.z_dim]).to(device)
alpha    = 3.0

img_orig = manipulate(G, z_sample, n_w_svm, 0, space='w')
img_z    = manipulate(G, z_sample, n_z_svm, alpha, space='z')
img_w    = manipulate(G, z_sample, n_w_svm, alpha, space='w')

fig, ax = plt.subplots(1, 3, figsize=(15, 5))
ax[0].imshow(img_orig); ax[0].set_title(f"Orijinal (Seed: {seed})");  ax[0].axis('off')
ax[1].imshow(img_z);    ax[1].set_title("Z-Space Manipülasyonu");       ax[1].axis('off')
ax[2].imshow(img_w);    ax[2].set_title("W-Space Manipülasyonu");       ax[2].axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# Cell 9 - Yöntem Karşılaştırması: SVM vs Mean Diff vs Score Weighted
import matplotlib.pyplot as plt

alpha = 4.0

img_svm   = manipulate(G, z_sample, n_w_svm,   alpha, space='w')
img_mean  = manipulate(G, z_sample, n_w_mean,  alpha, space='w')
img_score = manipulate(G, z_sample, n_w_score, alpha, space='w')  # yeni

fig, ax = plt.subplots(1, 4, figsize=(20, 5))
ax[0].imshow(img_orig);  ax[0].set_title("Orijinal")
ax[1].imshow(img_mean);  ax[1].set_title("Mean Difference")
ax[2].imshow(img_svm);   ax[2].set_title("LinearSVC")
ax[3].imshow(img_score); ax[3].set_title("Score Weighted (yeni)")
for a in ax:
    a.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# Cell 10 - Alpha Scaling (katman seçimli versiyon)

import matplotlib.pyplot as plt

alphas      = [-5, -2, 0, 2, 5]
mid_layers  = range(5, 12)   # yüz özellik katmanları

fig, axes = plt.subplots(2, len(alphas), figsize=(20, 8))

for i, a in enumerate(alphas):
    # Tüm katmanlar (orijinal davranış)
    img_all = manipulate(G, z_sample, n_w_svm, a, space='w', layer_range=None)
    # Sadece orta katmanlar
    img_mid = manipulate(G, z_sample, n_w_svm, a, space='w', layer_range=mid_layers)

    axes[0][i].imshow(img_all)
    axes[0][i].set_title(f"Tüm katmanlar\nα={a}")
    axes[0][i].axis('off')

    axes[1][i].imshow(img_mid)
    axes[1][i].set_title(f"Orta katmanlar (5–12)\nα={a}")
    axes[1][i].axis('off')

axes[0][0].set_ylabel("Tüm katmanlar", fontsize=12)
axes[1][0].set_ylabel("Orta katmanlar", fontsize=12)
plt.suptitle("Alpha Scaling — Tüm Katmanlar vs Seçici Katman Müdahalesi", y=1.02)
plt.tight_layout()
plt.show()